<a href="https://colab.research.google.com/github/ernestmwangombe/applied-ml-portfolio/blob/main/work/notebooks/freestyle_windows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup and Config

In [ ]:
%pip install -q duckdb huggingface_hub pandas numpy pyarrow     # install the tools (quietly)

import os                                    # files and folders
import numpy as np                           # maths helpers
import pandas as pd                          # tables
import duckdb                                # runs SQL on the data files
from google.colab import userdata            # Colab's secret store (where your token lives)
from huggingface_hub import HfApi, hf_hub_download   # list and download files from Hugging Face

HF_TOKEN = userdata.get("HF_TOKEN")          # read the secret token (never printed or typed here)
REPO = "FlyRank/internship-warehouse"        # the warehouse dataset

# ---------- CONFIG: the only block you change when new data arrives ----------
PRIOR_MONTHS  = ["2026-02"]                  # months that make up the "prior" window (features come from here)
FUTURE_MONTHS = ["2026-03"]                  # months that make up the "future" window (the outcome)
MIN_PRIOR_IMPR = 100                         # ignore pages with fewer prior impressions (too noisy)
DECLINE_RATIO  = 0.8                         # declined = future daily impressions below 80% of prior
CLIENT_DIP     = -0.10                       # a client "dipped" if its typical page fell 10% or more
MIN_CLIENT_PAGES = 20                        # need at least 20 eligible pages to trust a client's typical change
# June 2026 stays sealed: never put "2026-06" in either list while developing

2. The Reusable Window Builder

In [ ]:
api = HfApi(token=HF_TOKEN)                                              # connect to Hugging Face
ALL_FILES = api.list_repo_files(REPO, repo_type="dataset")               # list every file once

def fetch_months(months):                                                # download the daily-table files for some months
    paths = []                                                           # we will collect the local file paths here
    for m in months:                                                     # one month at a time
        wanted = [f for f in ALL_FILES
                  if f.startswith(f"fact_content_daily_performance/month={m}/")]   # files belonging to that month
        for f in wanted:                                                 # download each file (cached after the first time)
            paths.append(hf_hub_download(repo_id=REPO, repo_type="dataset", filename=f, token=HF_TOKEN))
    return paths                                                         # the list of local file paths

def sql_list(items):                                                     # turn ["a","b"] into 'a', 'b' for use in SQL
    return ", ".join("'" + x + "'" for x in items)

def build_windows(prior_months, future_months):                          # months in, one table of pages out
    months = prior_months + future_months                                # all the months we need
    paths = fetch_months(months)                                         # download them
    win_case = f"CASE WHEN month IN ({sql_list(prior_months)}) THEN 'prior' ELSE 'future' END"   # tag each row prior or future

    days = duckdb.sql(f"""
        SELECT {win_case} AS win, COUNT(DISTINCT report_date) AS n_days  -- calendar days in each window
        FROM read_parquet({paths}) WHERE month IN ({sql_list(months)}) GROUP BY win
    """).df().set_index("win")["n_days"]                                 # a small lookup: days per window

    long = duckdb.sql(f"""
        SELECT client_hash_id, content_hash_id, {win_case} AS win,       -- one row per page and window
               CAST(SUM(gsc_impressions) AS DOUBLE) AS impr              -- total impressions in that window
        FROM read_parquet({paths})
        WHERE gsc_data_available IS TRUE AND month IN ({sql_list(months)})   -- only rows with real search data
        GROUP BY client_hash_id, content_hash_id, win
    """).df()                                                            # run the query and get a table

    page = long.pivot_table(index=["client_hash_id", "content_hash_id"],
                            columns="win", values="impr", aggfunc="sum").fillna(0).reset_index()  # one row per page, with prior and future columns
    page["prior_daily"]  = page["prior"]  / days["prior"]                # average impressions per day, prior window
    page["future_daily"] = page["future"] / days["future"]               # average impressions per day, future window

    client = page.groupby("client_hash_id")[["prior_daily", "future_daily"]].sum()   # each client's total daily traffic
    client["client_total_change"] = (client["future_daily"] / client["prior_daily"]).replace([np.inf, -np.inf], np.nan) - 1   # its overall change
    page = page.merge(client[["client_total_change"]], left_on="client_hash_id", right_index=True, how="left")  # attach to each page

    page = page[page["prior"] >= MIN_PRIOR_IMPR].copy()                  # keep pages with enough prior impressions
    page["page_change"] = page["future_daily"] / page["prior_daily"] - 1 # this page's change (-0.25 means 25% fewer per day)
    grp = page.groupby("client_hash_id")["page_change"]                  # group pages by client
    page["client_median_change"] = grp.transform("median")               # the typical page's change in this client
    page.loc[grp.transform("size") < MIN_CLIENT_PAGES, "client_median_change"] = np.nan   # too few pages: do not trust it
    page["relative_change"] = page["page_change"] - page["client_median_change"]   # this page versus its client's typical page
    return page                                                          # the finished page table

3. Run It And Sort The Declines

In [ ]:
page = build_windows(PRIOR_MONTHS, FUTURE_MONTHS)                        # build the table for the configured months

page["declined"] = (page["future_daily"] < DECLINE_RATIO * page["prior_daily"]).astype(int)   # 1 if daily impressions fell below 80%
page["client_dipped"] = page["client_median_change"] <= CLIENT_DIP       # True if the client's typical page also fell
page["bucket"] = np.select(
    [page["declined"].eq(0),                                             # not declined
     page["declined"].eq(1) & page["client_dipped"],                     # declined and the client dipped too
     page["declined"].eq(1) & page["client_median_change"].notna()],     # declined while the client held steady
    ["not declined", "declined with client-wide dip", "declined, client steady"],
    default="declined, client unknown")                                  # declined, but too few pages to judge

print(f"Eligible pages: {len(page):,} | clients: {page['client_hash_id'].nunique()} | declined: {page['declined'].mean():.1%}")  # size and base rate
print(page["bucket"].value_counts().to_string())                         # how many pages fall in each bucket
print(page.groupby("bucket")["relative_change"].median().round(3).to_string())   # typical change versus the client's typical page

os.makedirs("work/outputs", exist_ok=True)                               # make the output folder in this session
page.to_parquet(f"work/outputs/windows_{PRIOR_MONTHS[0]}_{FUTURE_MONTHS[-1]}.parquet", index=False)  # cache the table (a data file: stays out of git)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Eligible pages: 80,322 | clients: 37 | declined: 26.7%
bucket
not declined                     58840
declined, client steady          17837
declined with client-wide dip     3609
declined, client unknown            36
bucket
declined with client-wide dip    0.000
declined, client steady         -0.501
declined, client unknown           NaN
not declined                     0.178


4. Did The Client Dip The Same Way Last Year?

In [ ]:
last = build_windows(["2025-02"], ["2025-03"])                           # the same Feb-to-Mar change, one year earlier
ly = last.groupby("client_hash_id")["client_total_change"].first().rename("client_change_last_year")  # each client's change that year
page = page.merge(ly, left_on="client_hash_id", right_index=True, how="left")   # attach it to this year's pages
page["usual_dip"] = page["client_change_last_year"] <= CLIENT_DIP        # True if the client also dipped Feb-to-Mar last year
print("Pages with last-year information:", page["client_change_last_year"].notna().sum())   # how many pages we could compare
print(page.groupby("bucket")["usual_dip"].agg(["size", "sum"]).to_string())   # per bucket: pages, and how many have a "usual dip" client

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Pages with last-year information: 20511
                                size  sum
bucket                                   
declined with client-wide dip   3609    0
declined, client steady        17837   79
declined, client unknown          36    0
not declined                   58840  764
